# Cointegration-Based Pairs Trading

**Strategy:**
- Entry: Z-score > 2.0 (short spread) or < -2.0 (long spread)
- Exit: Z-score crosses zero (mean reversion)
- Pairs: BTC/ETH, SOL/AVAX, or user-defined

In [ ]:
import sys

sys.path.append('..')

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

from medusa.features.cross_asset_features import cointegration_test, spread, z_score_spread
from medusa.strategies import PairsTradingStrategy

%matplotlib inline

In [ ]:
# Generate synthetic pair data with cointegration
np.random.seed(42)
n_days = 365 * 2
dates = pd.date_range('2024-01-01', periods=n_days, freq='D')

# Shared component (common trend)
common_trend = np.cumsum(np.random.randn(n_days)) + 100

# Asset 1: BTC
btc_specific = np.random.randn(n_days) * 2
btc_price = common_trend + btc_specific

# Asset 2: ETH (cointegrated with BTC via hedge ratio ~0.8)
hedge_ratio_true = 0.8
eth_specific = np.random.randn(n_days) * 1.5
eth_price = hedge_ratio_true * common_trend + eth_specific

# Create dual-asset DataFrame
data = pd.DataFrame({
    'close_BTC': btc_price,
    'close_ETH': eth_price,
}, index=dates)

print(f"Data shape: {data.shape}")
data.head()

In [ ]:
# Test cointegration
result = cointegration_test(data['close_BTC'], data['close_ETH'])
print("\nCointegration test results:")
print(f"  T-statistic: {result.t_stat:.4f}")
print(f"  P-value: {result.p_value:.4f}")
print(f"  Is cointegrated: {result.is_cointegrated}")
print(f"  Hedge ratio: {result.hedge_ratio:.4f}")
print(f"  True hedge ratio: {hedge_ratio_true}")

In [ ]:
# Calculate and visualize spread
spread_series = spread(data['close_BTC'], data['close_ETH'], result.hedge_ratio)
zscore = z_score_spread(spread_series, window=60)

fig, axes = plt.subplots(2, 1, figsize=(14, 8), sharex=True)

# Spread
axes[0].plot(spread_series.index, spread_series, label='Spread', color='blue')
axes[0].axhline(spread_series.mean(), color='red', linestyle='--', label='Mean')
axes[0].set_ylabel('Spread')
axes[0].set_title('Price Spread: BTC - β*ETH')
axes[0].legend()
axes[0].grid(True, alpha=0.3)

# Z-score
axes[1].plot(zscore.index, zscore, label='Z-score', color='purple')
axes[1].axhline(2.0, color='green', linestyle='--', label='Entry threshold (+2)')
axes[1].axhline(-2.0, color='green', linestyle='--', label='Entry threshold (-2)')
axes[1].axhline(0, color='red', linestyle='-', label='Exit (0)')
axes[1].set_ylabel('Z-score')
axes[1].set_xlabel('Date')
axes[1].set_title('Spread Z-score')
axes[1].legend()
axes[1].grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

In [ ]:
# Initialize strategy
strategy = PairsTradingStrategy(
    asset1='BTC',
    asset2='ETH',
    entry_zscore=2.0,
    exit_zscore=0.0,
    stop_zscore=3.0,
)

# Generate signals
entries, exits = strategy.generate_signals(data)

print(f"\nEntry signals: {entries.sum()}")
print(f"Exit signals: {exits.sum()}")

## Backtest Note

Pairs trading requires simultaneous long/short legs. For full backtest:
1. Construct synthetic 'spread' instrument
2. Run VectorBT on spread returns
3. OR use NautilusTrader for multi-leg execution

This notebook demonstrates signal generation. Full implementation in production.